In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
discharge_data = pd.read_csv('../../Data/Discharge-DHM/Bagmati at Khokana_River Water Level Inst_1990-01-01_2025-03-27_Point.csv')
flood_metadata_location = '../../Results/Floods/Metadata.csv'

In [6]:
# Identify top 10 flood events based on highest stage values
def identify_flood_events(discharge_data, num_floods=20, buffer_days=3, max_realistic_stage=10.0):
    """
    Identify top flood events based on peak stage values and create date ranges
    with buffer days before and after each peak.
    
    Parameters:
    discharge_data (DataFrame): DataFrame with 'dateTime' and 'value' columns
    num_floods (int): Number of top flood events to identify
    buffer_days (int): Number of days to extend before and after peak
    max_realistic_stage (float): Maximum realistic stage value in meters
    
    Returns:
    DataFrame: Flood events metadata with date ranges
    """
    
    # Clean and prepare the data
    df = discharge_data.copy()
    
    # Convert dateTime to datetime format
    df['dateTime'] = pd.to_datetime(df['dateTime'], format='%m/%d/%Y %H:%M:%S')
    
    # Remove missing or invalid values
    df = df.dropna(subset=['value'])
    df = df[df['value'] != '']  # Remove empty strings
    df = df[pd.to_numeric(df['value'], errors='coerce').notna()]  # Keep only numeric values
    df['value'] = pd.to_numeric(df['value'])
    
    # Filter out unrealistic values (sensor errors, etc.)
    initial_count = len(df)
    df = df[df['value'] >= 0]  # Remove negative values
    df = df[df['value'] <= max_realistic_stage]  # Remove unrealistically high values
    df = df[df['value'] != -9999990]  # Remove missing data flags
    
    print(f"📊 DATA CLEANING:")
    print(f"Initial records: {initial_count:,}")
    print(f"After cleaning: {len(df):,}")
    print(f"Removed: {initial_count - len(df):,} invalid/unrealistic records")
    
    # Sort by datetime
    df = df.sort_values('dateTime').reset_index(drop=True)
    
    print(f"\n📊 DATA OVERVIEW:")
    print(f"Date range: {df['dateTime'].min()} to {df['dateTime'].max()}")
    print(f"Stage value range: {df['value'].min():.3f} to {df['value'].max():.3f} m")
    
    # Find local maxima (peaks) to avoid selecting multiple points from the same flood event
    # We'll use a minimum separation of 7 days between peaks
    min_separation_days = 7
    
    # Sort by value to get highest peaks first
    df_sorted = df.sort_values('value', ascending=False).reset_index(drop=True)
    
    selected_peaks = []
    used_dates = []
    
    print(f"\n🔍 IDENTIFYING TOP {num_floods} FLOOD EVENTS:")
    print("-" * 70)
    
    for idx, row in df_sorted.iterrows():
        peak_date = row['dateTime']
        peak_value = row['value']
        
        # Check if this peak is far enough from already selected peaks
        is_separate = True
        for used_date in used_dates:
            if abs((peak_date - used_date).days) < min_separation_days:
                is_separate = False
                break
        
        if is_separate:
            selected_peaks.append({
                'peak_date': peak_date,
                'peak_value': peak_value,
                'peak_rank': len(selected_peaks) + 1
            })
            used_dates.append(peak_date)
            
            print(f"Flood {len(selected_peaks):2d}: {peak_date.strftime('%Y-%m-%d %H:%M')} | Stage: {peak_value:.3f} m")
            
            if len(selected_peaks) >= num_floods:
                break
    
    # Create flood events with extended date ranges
    flood_events = []
    
    print(f"\n📅 FLOOD EVENT DATE RANGES (±{buffer_days} days):")
    print("-" * 70)
    
    for i, peak in enumerate(selected_peaks, 1):
        # Calculate start and end dates with buffer
        start_date = peak['peak_date'] - pd.Timedelta(days=buffer_days)
        end_date = peak['peak_date'] + pd.Timedelta(days=buffer_days)
        
        # Calculate event duration
        duration_days = (end_date - start_date).days + 1
        
        flood_events.append({
            'flood_id': f"FLOOD_{i:02d}",
            'flood_rank': i,
            'peak_date': peak['peak_date'].strftime('%Y-%m-%d %H:%M:%S'),
            'peak_stage_m': peak['peak_value'],
            'start_date': start_date.strftime('%Y-%m-%d %H:%M:%S'),
            'end_date': end_date.strftime('%Y-%m-%d %H:%M:%S'),
            'duration_days': duration_days,
            'year': peak['peak_date'].year,
            'month': peak['peak_date'].month,
            'season': 'Monsoon' if peak['peak_date'].month in [6, 7, 8, 9] else 'Non-Monsoon'
        })
        
        print(f"  {flood_events[-1]['flood_id']}: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')} "
              f"(Peak: {peak['peak_value']:.3f} m on {peak['peak_date'].strftime('%Y-%m-%d')})")
    
    return pd.DataFrame(flood_events)

# Execute flood identification
print("🌊 IDENTIFYING FLOOD EVENTS FROM STAGE DATA")
print("=" * 80)

# Create Results/Floods directory if it doesn't exist
os.makedirs(os.path.dirname(flood_metadata_location), exist_ok=True)

# Identify flood events (limiting to realistic stage values ≤ 10m)
flood_metadata = identify_flood_events(discharge_data, num_floods=20, buffer_days=3, max_realistic_stage=10.0)

# Display the flood metadata
print(f"\n📋 FLOOD EVENTS METADATA:")
print("-" * 80)
print(flood_metadata[['flood_id', 'peak_date', 'peak_stage_m', 'start_date', 'end_date', 'season']].to_string(index=False))

# Save to CSV
flood_metadata.to_csv(flood_metadata_location, index=False)

print(f"\n💾 FLOOD METADATA SAVED:")
print(f"File: {flood_metadata_location}")
print(f"Records: {len(flood_metadata)} flood events")

# Summary statistics
print(f"\n📊 SUMMARY STATISTICS:")
print(f"Average peak stage: {flood_metadata['peak_stage_m'].mean():.3f} m")
print(f"Highest flood: {flood_metadata['peak_stage_m'].max():.3f} m ({flood_metadata.loc[flood_metadata['peak_stage_m'].idxmax(), 'peak_date']})")
print(f"Monsoon floods: {len(flood_metadata[flood_metadata['season'] == 'Monsoon'])}")
print(f"Non-monsoon floods: {len(flood_metadata[flood_metadata['season'] == 'Non-Monsoon'])}")
print(f"Years covered: {flood_metadata['year'].min()} - {flood_metadata['year'].max()}")

print(f"\n✅ TOP 10 FLOOD EVENTS IDENTIFIED AND SAVED!")

# Display first few rows of the saved CSV for verification
print(f"\n📄 SAVED FLOOD METADATA (first 5 rows):")
print("-" * 80)
saved_data = pd.read_csv(flood_metadata_location)
print(saved_data.head().to_string(index=False))

🌊 IDENTIFYING FLOOD EVENTS FROM STAGE DATA
📊 DATA CLEANING:
Initial records: 783,088
After cleaning: 780,583
Removed: 2,505 invalid/unrealistic records

📊 DATA OVERVIEW:
Date range: 2012-02-04 14:00:00 to 2024-09-28 05:40:00
Stage value range: 0.000 to 7.085 m

🔍 IDENTIFYING TOP 20 FLOOD EVENTS:
----------------------------------------------------------------------
Flood  1: 2024-04-21 12:50 | Stage: 7.085 m
Flood  2: 2018-07-04 17:30 | Stage: 6.356 m
Flood  3: 2024-09-28 05:30 | Stage: 6.165 m
Flood  4: 2019-04-16 16:35 | Stage: 5.600 m
Flood  5: 2024-07-31 05:40 | Stage: 5.591 m
Flood  6: 2024-07-06 16:40 | Stage: 5.099 m
Flood  7: 2023-08-08 09:20 | Stage: 5.034 m
Flood  8: 2022-08-10 02:40 | Stage: 4.736 m
Flood  9: 2019-07-12 21:00 | Stage: 4.731 m
Flood 10: 2021-09-06 07:30 | Stage: 4.707 m
Flood 11: 2012-10-11 07:05 | Stage: 4.414 m
Flood 12: 2020-07-20 17:50 | Stage: 4.199 m
Flood 13: 2024-07-23 03:40 | Stage: 4.093 m
Flood 14: 2022-06-15 03:00 | Stage: 4.070 m
Flood 15: 2015-0